In [5]:
import os, re, glob, time, subprocess, pythoncom, psutil, shutil, tempfile
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date
from pathlib import Path
from IPython.display import HTML, display
import polars as pl
import io

# ══════════════════════════════════════════════════════════════════════════════
# PATHS
# ══════════════════════════════════════════════════════════════════════════════
first_glob     = os.path.expanduser("~").replace("\\", "/")
CAPTURE_FOLDER = f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/CAPTURE/atd_realtime"
HC_PARQUET     = f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/hc_extend_combination.parquet"
MASTER_ROSTER  = f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Schedule/Schedule (Ops version)/2026/Master_Schedule_Merged.xlsx"
LEAVE_FILE     = f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/CAPTURE/leave.xlsx"
REPORT_SHIFT = "0500-1400"
DATE = "2026-08-26"

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
DISPLAY_NOTEBOOK = True
SEND_EMAIL       = False

EMAIL_TO = (
    "puneet.suneja@concentrix.com;"
    "kirpan.patar@concentrix.com;"
    "ML.HOC.Expedia.Hierarchy@concentrix.com;"
    "EG_CAI_RAYAH_expedia_global_rtm@concentrix.com"
)

EMAIL_CC = (
    "rahul.issar@concentrix.com;"
    "francesca.cioccari@concentrix.com;"
    "Varun.Kathuria@concentrix.com;"
    "urmila.chakka1@concentrix.com;"
    "VN_HOC_QUANG_vn_hcm_one_exp_wfm@concentrix.com;"
    "atul.pathak@concentrix.com"
)

# EMAIL_TO = ("huuchinh.nguyen@concentrix.com;")
# EMAIL_CC = ("huuchinh.nguyen@concentrix.com;")

TARGET_PLANNED   = 4.0
TARGET_UNPLANNED = 6.0
TARGET_SHRINKAGE = 10.0
TARGET_ATD       = 90.0

LEAVE_SHIFT_CODES = ["AL","CO","LWP","AB","SL","HAL","HLWP","HAB","HSL","CMLF","LATE"]
KEEP_LOBS         = ["Lodging","Non_Lodging"]

# ══════════════════════════════════════════════════════════════════════════════
# SHIFT HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def parse_shift_dt(shift_str, ref_date):
    if not shift_str or not isinstance(shift_str, str): return None
    parts = shift_str.strip().split("-")
    if len(parts) != 2: return None
    try:
        s = datetime.combine(ref_date, datetime.strptime(parts[0].zfill(4), "%H%M").time())
        e = datetime.combine(ref_date, datetime.strptime(parts[1].zfill(4), "%H%M").time())
        if e <= s: e += timedelta(days=1)
        return s, e
    except: return None

def is_night(shift_str):
    try: return int(str(shift_str).split("-")[0][:2]) >= 18
    except: return False

def get_report_scope():
    now    = datetime.now()
    today  = now.date()
    yester = today - timedelta(days=1)

    if REPORT_SHIFT:
        sh = REPORT_SHIFT
    else:
        try:
            _raw = pd.read_excel(MASTER_ROSTER, sheet_name="Sheet1", dtype=str)
            _raw.columns = [str(c) for c in _raw.columns]
            _info  = ["IEX ID","OracleID","Email","Employee Name","Status"]
            _dcols = [c for c in _raw.columns if re.match(r"\d{4}-\d{2}-\d{2}", c)]
            _long  = (_raw[_info + _dcols]
                      .melt(id_vars=_info, value_vars=_dcols,
                            var_name="Scheduled_Date", value_name="Roster_Shift"))
            _long["Scheduled_Date"] = pd.to_datetime(_long["Scheduled_Date"], errors="coerce").dt.date
            _today_shifts = set(
                _long[
                    (_long["Scheduled_Date"] == today) &
                    (_long["Status"] == "Active") &
                    (_long["Roster_Shift"].notna()) &
                    (~_long["Roster_Shift"].isin(["OFF","Termination","HO","null","nan",""]))
                ]["Roster_Shift"].dropna().unique()
            )
            available_shifts = sorted([
                s for s in _today_shifts
                if re.match(r"^\d{4}-\d{4}$", str(s))
            ], key=lambda s: int(s.split("-")[0]))
            print(f"Available shifts today: {available_shifts}")
        except Exception as e:
            print(f"Warning: Could not pre-load roster for shift detection: {e}")
            available_shifts = []

        cands = []
        for s in available_shifts:
            sh_h = int(s.split("-")[0][:2])
            ref  = yester if sh_h >= 18 else today
            p    = parse_shift_dt(s, ref)
            if p and p[0] <= now:
                cands.append((p[0], s))

        if not cands:
            day_shifts = [s for s in available_shifts if int(s.split("-")[0][:2]) < 18]
            sh = day_shifts[0] if day_shifts else "0600-1500"
            print(f"Warning: No shift started yet, using earliest: {sh}")
        else:
            sh = sorted(cands)[-1][1]

    sh_h  = int(sh.split("-")[0][:2])
    night = sh_h >= 18

    if DATE:
        roster_date = datetime.strptime(DATE, "%Y-%m-%d").date()
    else:
        roster_date = yester if night else today

    _avail = available_shifts if "available_shifts" in locals() else []

    if night:
        scope = list(_avail)
    else:
        rep_h = sh_h
        scope = [s for s in _avail
                 if int(s.split("-")[0][:2]) <= rep_h
                 and int(s.split("-")[0][:2]) < 18]

    scope = scope + LEAVE_SHIFT_CODES
    return sh, roster_date, scope

report_shift, roster_date, shifts_in_scope = get_report_scope()

report_now = datetime.strptime(DATE, "%Y-%m-%d")
report_date_s = report_now.strftime("%d-%b-%Y")
EMAIL_SUBJECT = f"Expedia VN Attendance Report as of the shift {report_shift} on {report_date_s} (VNT)"

print(f"Report shift   : {report_shift}")
print(f"Roster date    : {roster_date}")
print(f"Shifts in scope: {shifts_in_scope}")
print(f"Subject        : {EMAIL_SUBJECT}")

# ══════════════════════════════════════════════════════════════════════════════
# LOAD MASTER ROSTER
# ══════════════════════════════════════════════════════════════════════════════
print("Loading Master Schedule...")
raw = pd.read_excel(MASTER_ROSTER, sheet_name="Sheet1", dtype=str)
raw.columns = [str(c) for c in raw.columns]

info_cols = ["IEX ID","OracleID","Email","Employee Name","Status"]
date_cols = [c for c in raw.columns if re.match(r"\d{4}-\d{2}-\d{2}", c)]

roster_long = (
    raw[info_cols + date_cols]
    .melt(id_vars=info_cols, value_vars=date_cols,
          var_name="Scheduled_Date", value_name="Roster_Shift")
    .rename(columns={"IEX ID":"IEX","OracleID":"Emp ID","Employee Name":"Agent Name"})
)
roster_long["Scheduled_Date"] = pd.to_datetime(roster_long["Scheduled_Date"], errors="coerce").dt.date
roster_long["IEX"]    = pd.to_numeric(roster_long["IEX"],    errors="coerce")
roster_long["Emp ID"] = pd.to_numeric(roster_long["Emp ID"], errors="coerce")
roster_long["Roster_Shift"] = roster_long["Roster_Shift"].replace("-", np.nan)

def compute_roster_dt(df):
    rows_s, rows_e, nights = [], [], []
    for _, r in df.iterrows():
        ref = r["Scheduled_Date"]
        sh  = r["Roster_Shift"]
        p   = parse_shift_dt(sh, ref) if pd.notna(sh) else None
        rows_s.append(p[0] if p else pd.NaT)
        rows_e.append(p[1] if p else pd.NaT)
        nights.append(1 if pd.notna(sh) and is_night(sh) else 0)
    df = df.copy()
    df["Datetime_Start_Shift"] = rows_s
    df["Datetime_End_Shift"]   = rows_e
    df["Night_Shift"]          = nights
    return df

roster_long = compute_roster_dt(roster_long)

_actual_shifts = sorted(
    roster_long[
        (roster_long["Scheduled_Date"] == roster_date) &
        roster_long["Roster_Shift"].notna() &
        roster_long["Roster_Shift"].str.contains("-", na=False) &
        ~roster_long["Roster_Shift"].isin(["OFF","Termination","HO","null","nan",""])
    ]["Roster_Shift"].unique().tolist()
)
_sh_h  = int(report_shift.split("-")[0][:2])
_night = _sh_h >= 18
if _night:
    shifts_in_scope = _actual_shifts + LEAVE_SHIFT_CODES
else:
    shifts_in_scope = [
        s for s in _actual_shifts
        if int(s.split("-")[0][:2]) <= _sh_h
        and int(s.split("-")[0][:2]) < 18
    ] + LEAVE_SHIFT_CODES
print(f"Shifts in scope (from roster): {shifts_in_scope}")

today_roster = roster_long[
    (roster_long["Scheduled_Date"] == roster_date) &
    (~roster_long["Status"].isin(["Terminated",""])) &
    (roster_long["Roster_Shift"].notna()) &
    (~roster_long["Roster_Shift"].isin(["OFF","Termination","HO","null","nan",""])) &
    (
        roster_long["Roster_Shift"].isin(shifts_in_scope) |
        roster_long["Roster_Shift"].isin(LEAVE_SHIFT_CODES)
    )
].copy().reset_index(drop=True)

print(f"Roster agents in scope: {len(today_roster)}")

# Build IEX → email lookup for leave file mapping
iex_to_email = {}
for _, r in today_roster.iterrows():
    iex_val = str(int(r["IEX"])) if pd.notna(r.get("IEX")) else None
    email   = str(r["Email"]).strip().lower() if pd.notna(r.get("Email")) else None
    if iex_val and email:
        iex_to_email[iex_val] = email

# ══════════════════════════════════════════════════════════════════════════════
# LOAD HC EXTEND
# ══════════════════════════════════════════════════════════════════════════════
print("Loading HC Extend parquet...")
hc_pl = (
    pl.read_parquet(HC_PARQUET)
    .with_columns(pl.col("Date").cast(pl.Date, strict=False))
    .filter(pl.col("Date") <= pl.lit(roster_date))
    .sort("Date", descending=True)
    .unique(subset=["Email Id"], keep="first")
)
hc = hc_pl.to_pandas()
print(f"HC Extend rows (latest per agent <= {roster_date}): {len(hc)}")

email_col_hc = "Email Id"
if email_col_hc not in hc.columns:
    email_col_hc = next(
        (c for c in hc.columns if "email" in c.lower() and "supervisor" not in c.lower()), None)
    print(f"Fallback email col: {email_col_hc}")

HC_INFO_COLS = ["Supervisor Name","LOB","Wave","Alias","OracleID"]
if email_col_hc and not hc.empty:
    hc_map = (
        hc[[email_col_hc] + [c for c in HC_INFO_COLS if c in hc.columns]]
        .drop_duplicates(subset=[email_col_hc], keep="last")
        .rename(columns={email_col_hc: "_hc_email"})
    )
    hc_map["_hc_email"] = hc_map["_hc_email"].str.lower().str.strip()
else:
    hc_map = pd.DataFrame(columns=["_hc_email"] + HC_INFO_COLS)

print(f"HC map entries: {len(hc_map)}")

today_roster["_email_key"] = today_roster["Email"].str.lower().str.strip()
today_roster = today_roster.merge(
    hc_map.rename(columns={"_hc_email": "_email_key"}),
    on="_email_key", how="left"
)

if "OracleID_x" in today_roster.columns:
    today_roster["OracleID"] = today_roster["OracleID_x"].combine_first(today_roster["OracleID_y"])
    today_roster.drop(columns=["OracleID_x","OracleID_y"], inplace=True, errors="ignore")
elif "OracleID" not in today_roster.columns:
    today_roster["OracleID"] = today_roster["Emp ID"]

# ══════════════════════════════════════════════════════════════════════════════
# LOB MAPPING
# ══════════════════════════════════════════════════════════════════════════════
def map_lob(lob):
    if pd.isna(lob): return None
    s = str(lob).strip()
    if s == "Support_LG_Nesting": return "Lodging"
    if s == "Support_NL_Nesting": return "Non_Lodging"
    if s == "Lodging":            return "Lodging"
    if s == "Non_Lodging":        return "Non_Lodging"
    return None

if "LOB" in today_roster.columns:
    today_roster["LOB"] = today_roster["LOB"].apply(map_lob)

print(f"LOB distribution: {today_roster['LOB'].value_counts(dropna=False).to_dict()}")

# ══════════════════════════════════════════════════════════════════════════════
# LOAD LEAVE FILE — safe read even when file is open in Excel
# ══════════════════════════════════════════════════════════════════════════════
def load_leave_file(path):
    """Copy to temp dir first to bypass Excel file lock, then read."""
    try:
        tmp_dir  = tempfile.mkdtemp()
        tmp_path = os.path.join(tmp_dir, "leave_tmp.xlsx")
        shutil.copy2(path, tmp_path)
        df = pd.read_excel(tmp_path, dtype=str)
        print(f"Leave file loaded: {len(df)} rows")
        return df
    except Exception as e:
        print(f"Warning: Could not load leave file: {e}")
        return pd.DataFrame()

def build_leave_overrides(leave_df, roster_date, iex_to_email):
    if leave_df.empty: return {}

    leave_df = leave_df.copy()
    leave_df.columns = [str(c).strip() for c in leave_df.columns]

    date_col = next((c for c in leave_df.columns if "date"   in c.lower()), None)
    iex_col  = next((c for c in leave_df.columns if c.lower() == "iex"), None)
    lv_col   = next((c for c in leave_df.columns if "leave"  in c.lower()), None)
    half_col = next((c for c in leave_df.columns if "half"   in c.lower()), None)
    rsn_col  = next((c for c in leave_df.columns if "reason" in c.lower()), None)

    if not all([date_col, iex_col, lv_col]):
        print("Warning: Leave file missing required columns (Date, IEX, Leave)")
        return {}

    leave_df[date_col] = pd.to_datetime(leave_df[date_col], errors="coerce").dt.date
    leave_df = leave_df[leave_df[date_col] == roster_date].copy()
    print(f"Leave entries for {roster_date}: {len(leave_df)}")

    overrides = {}
    for _, row in leave_df.iterrows():
        try:
            iex_val = str(row[iex_col]).strip().split(".")[0]
            email   = iex_to_email.get(iex_val)
            if not email:
                print(f"  Warning: IEX {iex_val} not found in roster — skipped")
                continue

            code   = str(row[lv_col]).strip()   if pd.notna(row.get(lv_col))  else None
            half   = str(row[half_col]).strip() if half_col and pd.notna(row.get(half_col)) else None
            reason = str(row[rsn_col]).strip()  if rsn_col  and pd.notna(row.get(rsn_col))  else None

            # Normalize half: First → FHAL, Second → SHAL
            if half:
                half = ("FHAL" if half.lower().startswith("f")
                        else "SHAL" if half.lower().startswith("s") else None)

            if code:
                overrides[email.lower().strip()] = (code, half, reason, None)
        except Exception as ex:
            print(f"  Warning: Row processing error: {ex}")

    print(f"Leave overrides built: {len(overrides)} agent(s)")
    return overrides

print("Loading leave overrides from leave.xlsx...")
leave_raw_df    = load_leave_file(LEAVE_FILE)
LEAVE_OVERRIDES = build_leave_overrides(leave_raw_df, roster_date, iex_to_email)

# ══════════════════════════════════════════════════════════════════════════════
# APPLY LEAVE OVERRIDES
# ══════════════════════════════════════════════════════════════════════════════
def half_shift_str(roster_shift, half):
    p = parse_shift_dt(roster_shift, roster_date)
    if not p: return roster_shift
    s, e   = p
    half_h = (e - s).total_seconds() / 3600 / 2
    if half == "FHAL": ns, ne = s + timedelta(hours=half_h), e
    else:              ns, ne = s, s + timedelta(hours=half_h)
    return f"{ns.strftime('%H%M')}-{ne.strftime('%H%M')}"

today_roster["Leave"]       = None
today_roster["Final_Shift"] = today_roster["Roster_Shift"]
today_roster["Reason"]      = None
today_roster["Remark"]      = None
today_roster["Start_Shift"] = today_roster["Datetime_Start_Shift"]
today_roster["End_Shift"]   = today_roster["Datetime_End_Shift"]

for raw_email, ov in LEAVE_OVERRIDES.items():
    leave_code = ov[0]
    half       = ov[1] if len(ov) > 1 else None
    reason     = ov[2] if len(ov) > 2 else None
    remark     = ov[3] if len(ov) > 3 else None

    mask = today_roster["_email_key"] == raw_email.lower().strip()
    if not mask.any():
        print(f"  Warning: Leave override not found in roster: {raw_email}")
        continue

    today_roster.loc[mask, "Leave"]  = leave_code
    today_roster.loc[mask, "Reason"] = reason
    today_roster.loc[mask, "Remark"] = remark

    if leave_code in ("HAL","HLWP") and half in ("FHAL","SHAL"):
        today_roster.loc[mask, "Final_Shift"] = (
            today_roster.loc[mask, "Roster_Shift"]
            .apply(lambda s: half_shift_str(s, half))
        )
        for idx in today_roster[mask].index:
            p = parse_shift_dt(today_roster.at[idx, "Final_Shift"], roster_date)
            if p:
                today_roster.at[idx, "Start_Shift"] = p[0]
                today_roster.at[idx, "End_Shift"]   = p[1]
    elif leave_code in ("AL","CO","LWP","AB","SL","CMLF","NCNS"):
        today_roster.loc[mask, "Final_Shift"] = leave_code
        today_roster.loc[mask, "Start_Shift"] = pd.NaT
        today_roster.loc[mask, "End_Shift"]   = pd.NaT

print(f"Leave overrides applied: {today_roster['Leave'].notna().sum()} agent(s)")

# ══════════════════════════════════════════════════════════════════════════════
# LOGIN DATA
# ══════════════════════════════════════════════════════════════════════════════
print("Loading login CSVs...")

def load_min_logins():
    today     = roster_date
    cut_day   = datetime.combine(today, datetime.min.time())
    cut_night = datetime.combine(today, datetime.strptime("1800", "%H%M").time())
    yester    = today - timedelta(days=1)
    rows      = []

    for f in glob.glob(f"{CAPTURE_FOLDER}/*.csv"):
        try:
            stem  = Path(f).stem
            m     = re.search(r"(\w{3})\s+(\d{1,2})\s+(\d{4})", stem)
            if not m: continue
            fdate = pd.to_datetime(
                f"{m.group(1)} {m.group(2)} {m.group(3)}", format="%b %d %Y").date()
            if fdate not in (today, yester): continue

            tmp      = pd.read_csv(f, dtype=str, encoding="utf-8-sig")
            site_col = next((c for c in tmp.columns
                             if "business" in c.lower() or "location" in c.lower()), None)
            if site_col:
                tmp = tmp[tmp[site_col].str.contains("Ho Chi Minh", na=False, case=False)]

            ecol = next((c for c in tmp.columns if "email" in c.lower()), None)
            tcol = next((c for c in tmp.columns
                         if "login" in c.lower() and "time" in c.lower()), None)
            if not ecol or not tcol: continue

            tmp         = tmp[[ecol, tcol]].copy()
            tmp.columns = ["Agent Email","Login Time"]
            tmp["Login Time"]  = pd.to_datetime(tmp["Login Time"], errors="coerce")
            tmp["Agent Email"] = tmp["Agent Email"].str.lower().str.strip()
            rows.append(tmp.dropna(subset=["Login Time"]))
        except Exception as ex:
            print(f"  Warning: Skipping {Path(f).name}: {ex}")

    if not rows:
        empty = pd.DataFrame(columns=["Agent Email","Login Time"])
        return empty, empty

    raw       = pd.concat(rows, ignore_index=True)
    day_min   = (raw[raw["Login Time"] >= cut_day]
                 .groupby("Agent Email")["Login Time"].min().reset_index()
                 .rename(columns={"Login Time": "MinLoginTime"}))
    night_min = (raw[raw["Login Time"] >= cut_night]
                 .groupby("Agent Email")["Login Time"].min().reset_index()
                 .rename(columns={"Login Time": "MinLoginTime"}))
    return day_min, night_min

day_min_df, night_min_df = load_min_logins()
print(f"Day logins: {len(day_min_df)} | Night logins: {len(night_min_df)}")

def get_min_login(row):
    email = str(row.get("_email_key", ""))
    src   = night_min_df if is_night(str(row.get("Roster_Shift", ""))) else day_min_df
    m     = src[src["Agent Email"] == email]
    return m.iloc[0]["MinLoginTime"] if not m.empty else pd.NaT

today_roster["MinLoginTime"] = today_roster.apply(get_min_login, axis=1)
print(f"Agents with login: {today_roster['MinLoginTime'].notna().sum()}")

# ══════════════════════════════════════════════════════════════════════════════
# ATD CALCULATIONS
# ══════════════════════════════════════════════════════════════════════════════
LEAVE_DIRECT     = {"HAL","AL","LWP","AB","SL","CO","CMLF","LATE","NCNS"}
UNPLANNED_LEAVES = {"AB","SL","NCNS","CMLF"}
PLANNED_LEAVES   = {"AL","CO","LWP","HAL","HLWP","HAB","HSL","LATE"}

def _hrs(start, end):
    try:
        d = (pd.Timestamp(end) - pd.Timestamp(start)).total_seconds() / 3600
        return d if d > 0 else None
    except: return None

def calc_hc_schedule(row):
    sh = str(row.get("Roster_Shift", ""))
    if not sh or sh in ("nan","None",""): return None
    if any(x in sh for x in ["HAL","HAB","HLWP","HSL","In Training"]): return 1.0
    if sh in ("AL","CO","LWP","AB","SL","CMLF","LATE"): return 1.0
    h = _hrs(row.get("Datetime_Start_Shift"), row.get("Datetime_End_Shift"))
    if h is not None: return 1.0 if h > 6 else 0.5
    return None

def calc_attendance(row):
    shift = str(row.get("Final_Shift", ""))
    leave = row.get("Leave")
    login = row.get("MinLoginTime")
    if shift in LEAVE_DIRECT: return shift
    if leave and leave in LEAVE_DIRECT: return leave
    if pd.notna(login): return "PR"
    return "NCNS"

def calc_present(row):
    leave = row.get("Leave")
    shift = str(row.get("Final_Shift", ""))
    atd   = row.get("Attendance", "")
    tod   = _hrs(row.get("Start_Shift"), row.get("End_Shift"))
    if tod is None: return None
    if atd == "NCNS": return 0.0
    if leave is not None and tod < 6:  return 0.5
    if leave is not None and tod >= 6: return 0.0
    if "-" in shift and atd in ("HAL","HAB","HSL","HLWP"): return 0.5
    if "-" in shift and tod < 6:  return 0.5
    if "-" in shift and tod >= 6: return 1.0
    return None

def calc_planned(row):
    rs    = str(row.get("Roster_Shift", ""))
    leave = row.get("Leave")
    tod   = _hrs(row.get("Datetime_Start_Shift"), row.get("Datetime_End_Shift"))
    # Half-day leave codes in roster
    if any(x in rs for x in ["HAL","HLWP","HAB","HSL"]): return 0.5
    # Full-day planned leave codes directly in roster (AB excluded — AB is unplanned)
    if rs in ("AL","CO","SL","LWP","CMLF"):               return 1.0
    # Half-day leave override from leave.xlsx = unplanned, no planned contribution
    if leave in ("HAL","HLWP","HAB","HSL"):                return 0.0
    # Full-day planned leave override from leave.xlsx
    if leave in ("AL","CO","LWP"):                         return 1.0
    # Unplanned leave override
    if leave in ("AB","SL","CMLF","NCNS"):                 return 0.0
    if tod is None:
        p = parse_shift_dt(rs, roster_date)
        tod = (p[1]-p[0]).total_seconds()/3600 if p else None
    if tod is None: return None
    return 0.5 if tod < 5 else 0.0

def calc_unplanned(row):
    atd   = row.get("Attendance", "")
    leave = row.get("Leave")
    rs    = str(row.get("Roster_Shift", ""))
    # Half-day leave override = 0.5 unplanned
    if leave in ("HAL","HLWP","HAB","HSL"):                  return 0.5
    # Planned leave = no unplanned contribution
    if atd in PLANNED_LEAVES or leave in PLANNED_LEAVES:     return 0.0
    # Unplanned leave codes (from ATD or leave override)
    if atd in UNPLANNED_LEAVES or leave in UNPLANNED_LEAVES: return 1.0
    # AB/SL/CMLF directly in roster = unplanned
    if rs in ("AB","SL","CMLF"):                             return 1.0
    tod = _hrs(row.get("Start_Shift"), row.get("End_Shift"))
    if tod is None: return None
    if tod < 5  and atd != "PR": return 0.5
    if tod >= 5 and atd != "PR": return 1.0
    return 0.0

def calc_late(row):
    login = row.get("MinLoginTime")
    start = row.get("Start_Shift")
    if pd.isna(login) or pd.isna(start): return None
    diff_s = (pd.Timestamp(login) - pd.Timestamp(start)).total_seconds()
    return diff_s if diff_s > 180 else None

def fmt_hms(secs):
    if secs is None or (isinstance(secs, float) and np.isnan(secs)): return "00:00:00"
    s = int(abs(secs)); h, rem = divmod(s, 3600); m, ss = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{ss:02d}"

SHIFT_GROUP_ORDER = {"Morning": 0, "Mid": 1, "Night": 2, "Planned Leave": 3}

def get_shift_group(shift):
    if pd.isna(shift) or str(shift).strip() == "": return None
    s = str(shift).strip()
    if s in LEAVE_SHIFT_CODES: return "Planned Leave"
    if "-" not in s: return None
    try:
        h = int(s.split("-")[0][:2])
        if 5  <= h <= 8:  return "Morning"
        if 9  <= h <= 17: return "Mid"
        if h >= 18:       return "Night"
    except: pass
    return None

today_roster["HC Schedule"]  = today_roster.apply(calc_hc_schedule, axis=1)
today_roster["Attendance"]   = today_roster.apply(calc_attendance,  axis=1)
today_roster["Present"]      = today_roster.apply(calc_present,     axis=1)
today_roster["HC Planned"]   = today_roster.apply(calc_planned,     axis=1)
today_roster["HC Unplanned"] = today_roster.apply(calc_unplanned,   axis=1)
today_roster["Late_Secs"]    = today_roster.apply(calc_late,        axis=1)
today_roster["Late_Flag"]    = today_roster["Late_Secs"].notna().astype(int)
today_roster["Login Late"]   = today_roster["Late_Secs"].apply(fmt_hms)

atd_df = today_roster.copy()
atd_df["Shift_Group"] = atd_df["Roster_Shift"].apply(get_shift_group)

# ══════════════════════════════════════════════════════════════════════════════
# HC SCHEDULE DAY DENOMINATOR
# ══════════════════════════════════════════════════════════════════════════════
all_day_roster = roster_long[
    (roster_long["Scheduled_Date"] == roster_date) &
    (~roster_long["Status"].isin(["Terminated",""])) &
    (roster_long["Roster_Shift"].notna()) &
    (~roster_long["Roster_Shift"].isin(["OFF","Termination","HO","null","nan",""])) &
    (
        roster_long["Roster_Shift"].isin(shifts_in_scope) |
        roster_long["Roster_Shift"].isin(LEAVE_SHIFT_CODES)
    )
].copy()

all_day_roster["HC Schedule"] = all_day_roster.apply(calc_hc_schedule, axis=1)
all_day_roster["_email_key"]  = all_day_roster["Email"].str.lower().str.strip()
all_day_roster = all_day_roster.merge(
    hc_map.rename(columns={"_hc_email": "_email_key"}),
    on="_email_key", how="left"
)
if "OracleID_x" in all_day_roster.columns:
    all_day_roster.drop(columns=["OracleID_x","OracleID_y"], inplace=True, errors="ignore")
if "LOB" in all_day_roster.columns:
    all_day_roster["LOB"] = all_day_roster["LOB"].apply(map_lob)

hc_schedule_day = (
    all_day_roster[all_day_roster["LOB"].isin(KEEP_LOBS)]
    .groupby("LOB")["HC Schedule"].sum().reset_index()
    .rename(columns={"HC Schedule": "HC_Schedule_Day"})
)
hc_schedule_day_total = all_day_roster[
    all_day_roster["LOB"].isin(KEEP_LOBS)
]["HC Schedule"].sum()

_full_day = roster_long[
    (roster_long["Scheduled_Date"] == roster_date) &
    (~roster_long["Status"].isin(["Terminated",""])) &
    (roster_long["Roster_Shift"].notna()) &
    (~roster_long["Roster_Shift"].isin(["OFF","Termination","HO","null","nan",""]))
].copy()
_full_day["HC Schedule"] = _full_day.apply(calc_hc_schedule, axis=1)
_full_day["_email_key"]  = _full_day["Email"].str.lower().str.strip()
_full_day = _full_day.merge(hc_map.rename(columns={"_hc_email":"_email_key"}), on="_email_key", how="left")
if "LOB" in _full_day.columns:
    _full_day["LOB"] = _full_day["LOB"].apply(map_lob)

hc_full_day_by_lob = (
    _full_day[_full_day["LOB"].isin(KEEP_LOBS)]
    .groupby("LOB")["HC Schedule"].sum().reset_index()
    .rename(columns={"HC Schedule": "HC_Full_Day"})
)
hc_full_day_total = _full_day[_full_day["LOB"].isin(KEEP_LOBS)]["HC Schedule"].sum()

print(f"HC Full Day by LOB    : {hc_full_day_by_lob.to_dict('records')}")
print(f"HC Full Day Grand Total: {hc_full_day_total}")
print(f"HC Schedule Day by LOB    : {hc_schedule_day.to_dict('records')}")
print(f"HC Schedule Day Grand Total: {hc_schedule_day_total}")
print(f"ATD computed: {len(atd_df)} agents | "
      f"PR:{(atd_df['Attendance']=='PR').sum()} | "
      f"NCNS:{(atd_df['Attendance']=='NCNS').sum()} | "
      f"Leave:{atd_df['Attendance'].isin(list(LEAVE_DIRECT)).sum()}")

# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY BUILDERS
# ══════════════════════════════════════════════════════════════════════════════
def pct(n, d):
    try: return round(float(n)/float(d)*100, 2) if d and d > 0 else None
    except: return None

def build_summary(df, group_cols, schedule_override=None, shrinkage_denom=None):
    agg = {
        "Schedule"    : ("HC Schedule", "sum"),
        "Present"     : ("Present",      "sum"),
        "HC_Planned"  : ("HC Planned",   "sum"),
        "HC_Unplanned": ("HC Unplanned", "sum"),
        "Late"        : ("Late_Flag",    "sum"),
    }
    if group_cols:
        g = df.groupby(group_cols, dropna=False).agg(**agg).reset_index()
    else:
        g = pd.DataFrame([{k: df[v[0]].sum() for k, v in agg.items()}])

    def get_atd_denom(row):
        if schedule_override is None: return row["Schedule"]
        if isinstance(schedule_override, (int, float)): return schedule_override
        key = row.get("LOB") if "LOB" in row else None
        return schedule_override.get(key, row["Schedule"]) if key else row["Schedule"]

    def get_shr_denom(row):
        if shrinkage_denom is None: return row["Schedule"]
        if isinstance(shrinkage_denom, (int, float)): return shrinkage_denom
        key = row.get("LOB") if "LOB" in row else None
        return shrinkage_denom.get(key, row["Schedule"]) if key else row["Schedule"]

    g["Planned (%)"]    = g.apply(lambda r: pct(r["HC_Planned"],                   get_shr_denom(r)), axis=1)
    g["Unplanned (%)"]  = g.apply(lambda r: pct(r["HC_Unplanned"],                 get_shr_denom(r)), axis=1)
    g["Shrinkage (%)"]  = g.apply(lambda r: pct(r["HC_Planned"]+r["HC_Unplanned"], get_shr_denom(r)), axis=1)
    g["Attendance (%)"] = g.apply(
        lambda r: round(100 - pct(r["HC_Planned"] + r["HC_Unplanned"], get_shr_denom(r)), 2)
                  if pct(r["HC_Planned"] + r["HC_Unplanned"], get_shr_denom(r)) is not None
                  else None, axis=1)
    return g.rename(columns={"HC_Planned":"HC Planned","HC_Unplanned":"HC Unplanned"})

def grand_total(df, label_col, total_schedule_day=None, total_full_day=None):
    nc  = ["Schedule","Present","HC Planned","HC Unplanned","Late"]
    row = {label_col: "Grand Total"}
    for c in nc:
        if c in df.columns: row[c] = df[c].sum()
    atd_denom = total_schedule_day if total_schedule_day else row.get("Schedule", 0)
    shr_denom = total_full_day     if total_full_day     else atd_denom
    hs  = row.get("Schedule", 0)
    pl  = row.get("HC Planned", 0)
    ul  = row.get("HC Unplanned", 0)
    if shr_denom > 0:
        row["Planned (%)"]    = round(pl/shr_denom*100, 2)
        row["Unplanned (%)"]  = round(ul/shr_denom*100, 2)
        row["Shrinkage (%)"]  = round((pl+ul)/shr_denom*100, 2)
    if shr_denom > 0:
        shrinkage = (pl + ul) / shr_denom * 100
        row["Attendance (%)"] = round(100 - shrinkage, 2)
    return pd.concat([df, pd.DataFrame([row])], ignore_index=True)

atd_lob = atd_df[atd_df["LOB"].isin(KEEP_LOBS)].copy() if "LOB" in atd_df.columns else atd_df.copy()
lob_c   = "LOB" if "LOB" in atd_lob.columns else None
sup_c   = "Supervisor Name" if "Supervisor Name" in atd_lob.columns else None

sched_override_lob = dict(zip(hc_schedule_day["LOB"], hc_schedule_day["HC_Schedule_Day"]))

full_day_lob_dict = dict(zip(hc_full_day_by_lob["LOB"], hc_full_day_by_lob["HC_Full_Day"]))

# 1. Site wise
site_sum = build_summary(atd_lob, [lob_c] if lob_c else [],
                         schedule_override=sched_override_lob,
                         shrinkage_denom=full_day_lob_dict)
site_sum = grand_total(site_sum, lob_c or "LOB",
                       total_schedule_day=hc_schedule_day_total,
                       total_full_day=hc_full_day_total)

# 2. TL wise
tl_gc  = [c for c in [lob_c, sup_c] if c]
tl_sum = build_summary(atd_lob, tl_gc,
                       schedule_override=None,
                       shrinkage_denom=full_day_lob_dict) if tl_gc else pd.DataFrame()
if not tl_sum.empty:
    tl_sum = grand_total(tl_sum, tl_gc[-1],
                         total_schedule_day=hc_schedule_day_total,
                         total_full_day=hc_full_day_total)

# 3. Shift wise
sh_gc     = [c for c in [lob_c, "Roster_Shift"] if c and c in atd_lob.columns]
shift_sum = build_summary(atd_lob, sh_gc,
                          schedule_override=None,
                          shrinkage_denom=None) if sh_gc else pd.DataFrame()
if not shift_sum.empty:
    shift_sum = grand_total(shift_sum, sh_gc[-1],
                            total_schedule_day=hc_schedule_day_total,
                            total_full_day=hc_full_day_total)

# 4. Shift Group wise
sg_gc = [c for c in [lob_c, "Shift_Group"] if c and c in atd_lob.columns]
if sg_gc and "Shift_Group" in atd_lob.columns:
    shift_group_sum = build_summary(
        atd_lob[atd_lob["Shift_Group"].notna()], sg_gc,
        schedule_override=None,
        shrinkage_denom=None
    )
    shift_group_sum["_order"] = shift_group_sum["Shift_Group"].map(SHIFT_GROUP_ORDER).fillna(99)
    shift_group_sum = (shift_group_sum
                       .sort_values([lob_c or "LOB", "_order"])
                       .drop(columns=["_order"]).reset_index(drop=True))
    shift_group_sum = grand_total(shift_group_sum, sg_gc[-1],
                                  total_schedule_day=hc_schedule_day_total,
                                  total_full_day=hc_full_day_total)
else:
    shift_group_sum = pd.DataFrame()

# Absenteeism
def shift_sort_key(shift):
    if pd.isna(shift): return (1, 9999, "")
    s = str(shift)
    if "-" in s:
        try:    return (0, int(s.split("-")[0].zfill(4)), s)
        except: return (0, 9999, s)
    return (1, 0, s)

absent_df = atd_df[
    atd_df["Attendance"].isin(["NCNS","AB","SL","AL","LWP","HAL","CO","CMLF"]) &
    atd_df["LOB"].isin(KEEP_LOBS)
].copy()
absent_df["_sort_shift"] = absent_df["Roster_Shift"].apply(shift_sort_key)
sort_cols = [c for c in [lob_c, sup_c, "_sort_shift", "Agent Name"] if c and c in absent_df.columns]
if sort_cols: absent_df = absent_df.sort_values(sort_cols)
absent_df = absent_df.drop(columns=["_sort_shift"], errors="ignore")

def build_detail_xlsx(df):
    df = df.reset_index(drop=True)
    col_map = [
        ("OracleID",        "OracleID"),
        ("IEX ID",          "IEX"),
        ("Agent Name",      "Agent Name"),
        ("Email Id",        "Email"),
        ("Supervisor Name", "Supervisor Name"),
        ("LOB",             "LOB"),
        ("Alias",           "Alias"),
        ("Wave",            "Wave"),
        ("Shift",           "Roster_Shift"),
        ("Attendance",      "Attendance"),
    ]
    detail = pd.DataFrame()
    for out_col, src_col in col_map:
        detail[out_col] = df[src_col].values if src_col in df.columns else None

    detail["Login Time"] = df["MinLoginTime"].apply(
        lambda x: x.strftime("%Y-%m-%d %H:%M:%S") if pd.notna(x) else ""
    )
    detail["Late (mins)"] = df["Late_Secs"].apply(
        lambda x: round(x / 60, 1) if pd.notna(x) and not np.isnan(x) else ""
    )
    detail["Late (Count)"] = df["Late_Flag"].fillna(0).astype(int)
    detail["Reason"]       = df["Reason"].fillna("") if "Reason" in df.columns else ""

    sort_cols = [c for c in ["Shift"]
                 if c in detail.columns]
    if sort_cols:
        detail = detail.sort_values(sort_cols, na_position="last").reset_index(drop=True)

    buf = io.BytesIO()
    with pd.ExcelWriter(buf, engine="openpyxl") as writer:
        detail.to_excel(writer, sheet_name="ATD Detail", index=False)
        ws = writer.sheets["ATD Detail"]
        # Style header
        from openpyxl.styles import PatternFill, Font, Alignment
        hdr_fill = PatternFill("solid", fgColor="1A3A5C")
        for cell in ws[1]:
            cell.fill      = hdr_fill
            cell.font      = Font(color="FFFFFF", bold=True)
            cell.alignment = Alignment(horizontal="center")
        # Auto column width
        for col_cells in ws.columns:
            max_len = max((len(str(c.value or "")) for c in col_cells), default=8)
            ws.column_dimensions[col_cells[0].column_letter].width = min(max_len + 3, 45)
    buf.seek(0)
    return buf

_detail_buf  = build_detail_xlsx(atd_df[atd_df["LOB"].isin(KEEP_LOBS)].copy())
_detail_path = os.path.join(
    tempfile.gettempdir(),
    f"ATD_Realtime_Detail_{roster_date}_{report_shift.replace('-','_')}.xlsx"
)
with open(_detail_path, "wb") as _f:
    _f.write(_detail_buf.read())
print(f"Detail xlsx saved: {_detail_path}")

print(f"Site:{len(site_sum)} | TL:{len(tl_sum)} | Shift:{len(shift_sum)} | "
      f"ShiftGroup:{len(shift_group_sum)} | Absent:{len(absent_df)}")

# ══════════════════════════════════════════════════════════════════════════════
# STYLE CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════
MET_BG="#d4f4e2"; MET_FG="#1a5c2a"
MISS_BG="#fde8ea"; MISS_FG="#9b1c2a"
WARN_BG="#fff3cd"; WARN_FG="#7a5200"
HDR_DARK="#1a3a5c"; HDR_MID="#1f5c99"
TOT_BG="#1a3a5c"
WHT_ROW="#ffffff"
BANNER_C="#8b0020"
SEC_BADGE_BG="#e6a817"
SEC_BADGE_FG="#1a1a1a"
FONT="font-family:Arial,sans-serif;font-size:11px;"
TH_S=(f"{FONT}padding:5px 8px;color:#fff;font-weight:bold;"
      f"white-space:nowrap;text-align:center;"
      f"border:1px solid rgba(255,255,255,0.2);")
TD_S=f"{FONT}padding:4px 8px;border:1px solid #e8e8e8;white-space:nowrap;"
TD_TOT=(f"{FONT}padding:4px 8px;border:1px solid rgba(255,255,255,0.15);"
        f"background:{TOT_BG};color:#fff;font-weight:bold;")

SHIFT_GROUP_STYLE = {
    "Morning":       {"bg": "#2E7D32", "fg": "#ffffff"},
    "Mid":           {"bg": "#E65100", "fg": "#ffffff"},
    "Night":         {"bg": "#1565C0", "fg": "#ffffff"},
    "Planned Leave": {"bg": "#757575", "fg": "#ffffff"},
}

CSS=f"""
body{{margin:0;padding:16px;background:#fff;font-family:Arial,sans-serif}}
.t{{border-collapse:collapse;font-size:11px;white-space:nowrap;width:auto}}
.t thead th{{padding:5px 8px;color:#fff;font-weight:bold;text-align:center;border:1px solid rgba(255,255,255,0.2)}}
.t tbody td{{padding:4px 8px;border:1px solid #e8e8e8;text-align:left;background:#fff}}
.t tbody tr.tot td{{background:{TOT_BG}!important;color:#fff!important;font-weight:bold!important}}
.met{{background:{MET_BG}!important;color:{MET_FG}!important;font-weight:bold!important}}
.miss{{background:{MISS_BG}!important;color:{MISS_FG}!important;font-weight:bold!important}}
.sec-badge{{display:inline-block;font-size:12px;font-weight:bold;background:{SEC_BADGE_BG};color:{SEC_BADGE_FG};padding:4px 12px;margin:24px 0 4px;border-radius:3px}}
.note{{font-size:10.5px;color:#555;background:#f8f8f8;border-left:3px solid {SEC_BADGE_BG};padding:4px 10px;margin:0 0 10px;border-radius:0 3px 3px 0}}
"""

# ══════════════════════════════════════════════════════════════════════════════
# HTML HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def fv(v, p=False):
    if v is None or (isinstance(v, float) and np.isnan(v)): return "&#8212;"
    if p: return f"{float(v):.2f}%"
    if isinstance(v, float): return f"{v:,.1f}"
    return str(v)

def _cc(v, tgt, hb=True, em=False):
    if v is None or (isinstance(v, float) and np.isnan(v)): return ("","")
    met = float(v) <= tgt if hb else float(v) >= tgt
    if em:
        s = (f"background:{MET_BG};color:{MET_FG};font-weight:bold;" if met
             else f"background:{MISS_BG};color:{MISS_FG};font-weight:bold;")
        return ("", s)
    return ("met" if met else "miss", "")

def _th(l, bg=HDR_MID):   return f'<th style="{TH_S}background:{bg};">{l}</th>'
def _thl(l, bg=HDR_DARK): return f'<th style="{TH_S}background:{bg};text-align:left;">{l}</th>'

def _tdl(v, tot=False, em=False, bg=WHT_ROW):
    val = str(v) if v is not None and not (isinstance(v,float) and np.isnan(v)) else "&#8212;"
    if tot: return f'<td style="{TD_TOT}text-align:left;">{val}</td>'
    return f'<td style="{TD_S}background:#fff;">{val}</td>'

def _tdn(v, tot=False, em=False, bg=WHT_ROW):
    val = fv(v)
    if tot: return f'<td style="{TD_TOT}text-align:right;">{val}</td>'
    return f'<td style="{TD_S}text-align:right;background:#fff;">{val}</td>'

def _tdp(v, tgt, hb=True, tot=False, em=False, bg=WHT_ROW):
    val = fv(v, True)
    if tot: return f'<td style="{TD_TOT}text-align:right;">{val}</td>'
    cls, inl = _cc(v, tgt, hb, em)
    if em:  return f'<td style="{TD_S}text-align:right;background:#fff;{inl}">{val}</td>'
    return  f'<td class="{cls}" style="{TD_S}text-align:right;background:#fff;">{val}</td>'

def _sec(num, title, note, em=False):
    bs = (f"display:inline-block;font-size:12px;font-weight:bold;"
          f"background:{SEC_BADGE_BG};color:{SEC_BADGE_FG};"
          f"padding:4px 12px;margin:24px 0 4px;border-radius:3px;")
    ns = (f"{FONT}font-size:10.5px;color:#555;background:#f8f8f8;"
          f"border-left:3px solid {SEC_BADGE_BG};"
          f"padding:4px 10px;margin:0 0 10px;"
          f"border-radius:0 3px 3px 0;display:block;")
    if em:
        return (f'<p style="margin:24px 0 4px;">'
                f'<span style="{bs}">{num}. {title}</span></p>'
                f'<p style="{ns}">{note}</p>')
    return (f'<div style="margin:24px 0 4px;">'
            f'<span class="sec-badge" style="background:{SEC_BADGE_BG};color:{SEC_BADGE_FG};">'
            f'{num}. {title}</span></div>'
            f'<div class="note" style="border-left-color:{SEC_BADGE_BG};">{note}</div>')

SITE_SPEC = [
    ("LOB",           "LOB",            True,  None,             True),
    ("Schedule",      "Schedule",       False, None,             True),
    ("Present",       "Present",        False, None,             True),
    ("HC Planned",    "HC Planned",     False, None,             True),
    ("HC Unplanned",  "HC Unplanned",   False, None,             True),
    ("Late",          "Late",           False, None,             True),
    ("Planned (%)",   "Planned (%)",    False, TARGET_PLANNED,   True),
    ("Unplanned (%)","Unplanned (%)",   False, TARGET_UNPLANNED, True),
    ("Shrinkage (%)","Shrinkage (%)",   False, TARGET_SHRINKAGE, True),
    ("Attendance (%)","Attendance (%)", False, TARGET_ATD,       False),
]
TL_SPEC = [
    ("LOB",           "LOB",            True,  None,             True),
    ("Supervisor",    "Supervisor Name",True,  None,             True),
    ("Schedule",      "Schedule",       False, None,             True),
    ("Present",       "Present",        False, None,             True),
    ("HC Planned",    "HC Planned",     False, None,             True),
    ("HC Unplanned",  "HC Unplanned",   False, None,             True),
    ("Late",          "Late",           False, None,             True),
    ("Login Late",    "Login Late",     True,  None,             True),
    ("Planned (%)",   "Planned (%)",    False, TARGET_PLANNED,   True),
    ("Unplanned (%)","Unplanned (%)",   False, TARGET_UNPLANNED, True),
    ("Shrinkage (%)","Shrinkage (%)",   False, TARGET_SHRINKAGE, True),
    ("Attendance (%)","Attendance (%)", False, TARGET_ATD,       False),
]
SH_SPEC = [
    ("LOB",           "LOB",            True,  None,             True),
    ("Roster Shift",  "Roster_Shift",   True,  None,             True),
    ("Schedule",      "Schedule",       False, None,             True),
    ("Present",       "Present",        False, None,             True),
    ("HC Planned",    "HC Planned",     False, None,             True),
    ("HC Unplanned",  "HC Unplanned",   False, None,             True),
    ("Late",          "Late",           False, None,             True),
    ("Login Late",    "Login Late",     True,  None,             True),
    ("Planned (%)",   "Planned (%)",    False, TARGET_PLANNED,   True),
    ("Unplanned (%)","Unplanned (%)",   False, TARGET_UNPLANNED, True),
    ("Shrinkage (%)","Shrinkage (%)",   False, TARGET_SHRINKAGE, True),
    ("Attendance (%)","Attendance (%)", False, TARGET_ATD,       False),
]
SG_SPEC = [
    ("LOB",           "LOB",            True,  None,             True),
    ("Shift Group",   "Shift_Group",    True,  None,             True),
    ("Schedule",      "Schedule",       False, None,             True),
    ("Present",       "Present",        False, None,             True),
    ("HC Planned",    "HC Planned",     False, None,             True),
    ("HC Unplanned",  "HC Unplanned",   False, None,             True),
    ("Late",          "Late",           False, None,             True),
    ("Planned (%)",   "Planned (%)",    False, TARGET_PLANNED,   True),
    ("Unplanned (%)","Unplanned (%)",   False, TARGET_UNPLANNED, True),
    ("Shrinkage (%)","Shrinkage (%)",   False, TARGET_SHRINKAGE, True),
    ("Attendance (%)","Attendance (%)", False, TARGET_ATD,       False),
]

# ══════════════════════════════════════════════════════════════════════════════
# TABLE RENDERERS
# ══════════════════════════════════════════════════════════════════════════════
def render_table(df, spec, em=False):
    tc  = "" if em else 'class="t" '
    col = [(h, k, il, tgt, hb) for h, k, il, tgt, hb in spec if k in df.columns]
    h   = [f'<table {tc}style="border-collapse:collapse;width:auto;{FONT}"><thead><tr>']
    for hdr, _, il, _t, _h in col:
        h.append(_thl(hdr) if il else _th(hdr))
    h.append('</tr></thead><tbody>')

    for i, row in df.iterrows():
        tot = any(str(row.get(c, "")) == "Grand Total"
                  for c in ["LOB","Supervisor Name","Roster_Shift","Shift_Group"]
                  if c in df.columns)
        h.append('<tr>' if em else f'<tr class="{"tot" if tot else ""}">')
        for _, k, il, tgt, hb in col:
            v = row.get(k)
            if k == "Shift_Group" and not tot:
                s = SHIFT_GROUP_STYLE.get(str(v) if v else "", None)
                if s:
                    h.append(f'<td style="{TD_S}background:{s["bg"]};color:{s["fg"]};'
                             f'font-weight:bold;text-align:center;">'
                             f'{v if v else "&#8212;"}</td>')
                else:
                    h.append(_tdl(v, tot, em))
            elif il:    h.append(_tdl(v, tot, em))
            elif tgt:   h.append(_tdp(v, tgt, hb, tot, em))
            else:       h.append(_tdn(v, tot, em))
        h.append('</tr>')
    h.append('</tbody></table>')
    return "".join(h)

def render_absent(df, em=False):
    ACOLORS = {
        "NCNS": (MISS_BG,  MISS_FG),
        "AB":   (MISS_BG,  MISS_FG),
        "HAL":  (WARN_BG,  WARN_FG),
        "AL":   ("#dce8f5","#1a3a5c"),
        "SL":   ("#e8f4fd","#1a3a5c"),
        "CO":   ("#e8f5e9","#1a5c2a"),
        "LWP":  ("#f3e5f5","#6a1b9a"),
        "CMLF": (MISS_BG,  MISS_FG),
    }
    show = ["OracleID", "IEX", "Agent Name","Supervisor Name","LOB",
            "Alias","Wave","Roster_Shift","Attendance","Reason"]
    cols = [c for c in show if c in df.columns]
    lbls = {"Roster_Shift": "Shift", "IEX": "IEX ID"}
    tc   = "" if em else 'class="t" '
    h    = [f'<table {tc}style="border-collapse:collapse;width:auto;{FONT}"><thead><tr>']
    for c in cols:
        h.append(_thl(lbls.get(c, c)))
    h.append('</tr></thead><tbody>')

    for i, row in df.iterrows():
        atd    = str(row.get("Attendance", ""))
        ba, fa = ACOLORS.get(atd, ("#fff","#000"))
        h.append('<tr>')
        for c in cols:
            v   = row.get(c, "")
            val = (str(v) if v is not None
                   and not (isinstance(v, float) and np.isnan(v)) else "&#8212;")
            if c == "Attendance":
                h.append(f'<td style="{TD_S}background:{ba};color:{fa};'
                         f'font-weight:bold;text-align:center;">{val}</td>')
            else:
                h.append(f'<td style="{TD_S}background:#fff;">{val}</td>')
        h.append('</tr>')
    h.append('</tbody></table>')
    return "".join(h)

# ══════════════════════════════════════════════════════════════════════════════
# BUILD ALL SECTIONS
# ══════════════════════════════════════════════════════════════════════════════
def build_all(em=False):
    parts = []
    bi  = f"{FONT}font-size:14px;font-weight:bold;color:#fff;margin:0;"
    bs  = f"{FONT}font-size:10.5px;color:#fff;margin:3px 0 0;"
    inn = (f'<p style="{bi}">&#128202; Expedia VN Attendance Report — Real-time</p>'
           f'<p style="{bs}">Shift: <strong>{report_shift}</strong> &nbsp;|&nbsp; '
           f'Roster: {roster_date} &nbsp;|&nbsp; '
           f'Generated: {report_now.strftime("%Y-%m-%d %H:%M")} &nbsp;|&nbsp; '
           f'Targets: Planned &le;{TARGET_PLANNED:.0f}% / '
           f'Unplanned &le;{TARGET_UNPLANNED:.0f}% / '
           f'Atd &ge;{TARGET_ATD:.0f}%</p>')
    if em:
        parts.append(f'<table width="100%" border="0" cellspacing="0" cellpadding="0" style="margin:0 0 14px;">'
                     f'<tr><td style="background:{BANNER_C};padding:10px 14px;border-radius:4px;">'
                     f'{inn}</td></tr></table>')
    else:
        parts.append(f'<div style="background:{BANNER_C};padding:10px 14px;'
                     f'border-radius:4px;margin:0 0 14px;">{inn}</div>')

    def spacer():
        if em:
            return ('<table width="100%" border="0" cellspacing="0" cellpadding="0">'
                    '<tr><td style="height:28px;font-size:1px;line-height:1px;">&nbsp;</td></tr></table>')
        return '<div style="height:28px;"></div>'

    parts.append(_sec("1","Site wise",
                      f"Overall summary — shift {report_shift} on {roster_date}.", em))
    parts.append(render_table(site_sum, SITE_SPEC, em))

    parts.append(spacer())
    parts.append(_sec("2","TL wise",
                      "By Team Leader. Login Late = time past shift start (grace 3 min).", em))
    parts.append(render_table(tl_sum, TL_SPEC, em) if not tl_sum.empty
                 else '<p style="color:#888;font-size:11px;">No data</p>')

    parts.append(spacer())
    parts.append(_sec("3","Shift wise",
                      f"By roster shift for {roster_date}.", em))
    parts.append(render_table(shift_sum, SH_SPEC, em) if not shift_sum.empty
                 else '<p style="color:#888;font-size:11px;">No data</p>')

    parts.append(spacer())
    parts.append(_sec("4","Shift Group wise",
                      f"Morning (05–08h) · Mid (09–17h) · Night (18h+) — {roster_date}.", em))
    parts.append(render_table(shift_group_sum, SG_SPEC, em) if not shift_group_sum.empty
                 else '<p style="color:#888;font-size:11px;">No data</p>')

    parts.append(spacer())
    parts.append(_sec("5","Absenteeism",
                      f"Agents absent/on-leave for {roster_date}. "
                      f"Total: <strong>{len(absent_df)}</strong>. "
                      f"Sorted: LOB → Supervisor → Shift → Agent.", em))
    parts.append(render_absent(absent_df, em) if not absent_df.empty
                 else f'<p style="color:{MET_FG};font-size:11px;">&#9989; No absenteeism.</p>')

    return "".join(parts)

# ══════════════════════════════════════════════════════════════════════════════
# EMAIL GREETING / SIGNATURE
# ══════════════════════════════════════════════════════════════════════════════
def greeting():
    return f"""
<p style="{FONT}font-size:12px;margin:0 0 10px;line-height:1.7">Dear team,</p>
<p style="{FONT}font-size:12px;margin:0 0 10px;line-height:1.7">
    Please find the Expedia VN Attendance Report till the shift
    <strong>{report_shift}</strong> on <strong>{report_date_s}</strong> (VNT).
</p>
<hr style="border:none;border-top:1px solid #e0e0e0;margin:0 0 12px;">
"""

def signature():
    return f"""
<hr style="border:none;border-top:1px solid #e0e0e0;margin:12px 0 10px;">
<p style="{FONT}font-size:12px;margin:0 0 4px;">Thanks &amp; Regards,</p>
<p style="{FONT}font-size:12px;font-weight:bold;margin:0 0 2px;">Chinh Nguyen</p>
<p style="{FONT}font-size:11px;color:#555;font-weight:bold;margin:0 0 2px;">Analyst, WFM Real Time Management</p>
<p style="{FONT}font-size:11px;color:#555;margin:0 0 2px;line-height:1.6">
    Level 4, Tower 1, OneHub Saigon, Lot C1-2, D1 Street, Saigon Hi Tech Park,<br>
    Tan Phu Ward, District 9, Ho Chi Minh City, Vietnam
</p>
<p style="{FONT}font-size:11px;color:#555;margin:0;">
    Ph No: +84 986 473 419 &nbsp;|&nbsp;
    Email: <a href="mailto:huuchinh.nguyen@concentrix.com"
       style="color:{HDR_MID};font-weight:bold;text-decoration:none;">
       huuchinh.nguyen@concentrix.com</a>
</p>
<p style="{FONT}font-size:10px;color:#aaa;margin-top:8px;">
    Generated: {report_now.strftime("%Y-%m-%d %H:%M")} &nbsp;|&nbsp;
    Source: Master Schedule + hc_extend_combination.parquet + CAPTURE/atd_realtime
</p>
"""

# ══════════════════════════════════════════════════════════════════════════════
# DISPLAY
# ══════════════════════════════════════════════════════════════════════════════
if DISPLAY_NOTEBOOK:
    nb  = ("<!DOCTYPE html><html><head><meta charset='utf-8'>"
           f"<style>{CSS}</style></head><body>"
           + build_all(em=False) + "</body></html>")
    esc = nb.replace("&","&amp;").replace('"',"&quot;").replace("'","&#39;")
    display(HTML(
        f'<iframe srcdoc="{esc}" style="width:100%;border:none;min-height:800px;" '
        f'onload="this.style.height=(this.contentDocument.body.scrollHeight+40)+\'px\'"></iframe>'))
    print("Display done")

# ══════════════════════════════════════════════════════════════════════════════
# SEND EMAIL
# ══════════════════════════════════════════════════════════════════════════════
if SEND_EMAIL:
    import win32com.client

    html_body = (
        "<!--[if mso]><xml><o:OfficeDocumentSettings><o:AllowPNG/>"
        "<o:PixelsPerInch>96</o:PixelsPerInch></o:OfficeDocumentSettings></xml><![endif]-->"
        f"<div style='padding:20px 24px;background:#fff;{FONT}'>"
        + greeting() + build_all(em=True) + signature() + "</div>"
    )

    def send_or_reply(to, cc, subj, body, quit_after=True):
        pythoncom.CoInitialize()
        was_on = any(p.name().lower() == "outlook.exe"
                    for p in psutil.process_iter(["name"]))
        if not was_on:
            for exe in [
                r"C:\Program Files\Microsoft Office\root\Office16\OUTLOOK.EXE",
                r"C:\Program Files (x86)\Microsoft Office\root\Office16\OUTLOOK.EXE",
            ]:
                if os.path.exists(exe): subprocess.Popen([exe]); break
            print("Starting Outlook...")
            for _ in range(30):
                time.sleep(1)
                try: win32com.client.GetActiveObject("Outlook.Application"); break
                except: pass
        try:
            ol    = win32com.client.Dispatch("Outlook.Application")
            ns    = ol.GetNamespace("MAPI"); ns.Logon()
            print("Sending new email...")
            mail = ol.CreateItem(0)
            mail.To = to; mail.CC = cc
            mail.Subject = subj; mail.HTMLBody = body
            if os.path.exists(_detail_path):
                mail.Attachments.Add(_detail_path)
                print(f"Attachment added: {os.path.basename(_detail_path)}")
            mail.Send()

            print(f"Email sent to: {to}")
            time.sleep(3)
        finally:
            if quit_after and not was_on:
                try: ol.Quit(); print("Outlook closed")
                except: pass

    send_or_reply(EMAIL_TO, EMAIL_CC, EMAIL_SUBJECT, html_body)

Report shift   : 0500-1400
Roster date    : 2026-08-26
Shifts in scope: ['AL', 'CO', 'LWP', 'AB', 'SL', 'HAL', 'HLWP', 'HAB', 'HSL', 'CMLF', 'LATE']
Subject        : Expedia VN Attendance Report as of the shift 0500-1400 on 26-Aug-2026 (VNT)
Loading Master Schedule...
Shifts in scope (from roster): ['0500-1400', 'AL', 'CO', 'LWP', 'AB', 'SL', 'HAL', 'HLWP', 'HAB', 'HSL', 'CMLF', 'LATE']
Roster agents in scope: 18
Loading HC Extend parquet...
HC Extend rows (latest per agent <= 2026-08-26): 894
HC map entries: 894
LOB distribution: {'Lodging': 15, None: 3}
Loading leave overrides from leave.xlsx...
Leave file loaded: 879 rows
Leave entries for 2026-08-26: 0
Leave overrides built: 0 agent(s)
Leave overrides applied: 0 agent(s)
Loading login CSVs...
Day logins: 12 | Night logins: 0
Agents with login: 12
HC Full Day by LOB    : [{'LOB': 'Lodging', 'HC_Full_Day': 78.0}]
HC Full Day Grand Total: 78.0
HC Schedule Day by LOB    : [{'LOB': 'Lodging', 'HC_Schedule_Day': 15.0}]
HC Schedule Day Gr

c:\Users\huuchinh.nguyen\AppData\Local\anaconda3\Lib\site-packages\IPython\core\display.py:431: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


Display done


In [6]:
leave_raw_df

,Date,IEX,Leave,Half Shift,Agent Name,Reason
0,2026-02-27 00:00:00,3087699,AB,NaN,TRAN BA PHUC,Unknown reason
1,2026-02-27 00:00:00,3112765,AB,NaN,DANG CHAU ANH,Unknown reason
2,2026-02-27 00:00:00,3109420,AB,NaN,NGUYEN BUI THANH TRUC,Unknown reason
3,2026-02-27 00:00:00,3112719,HAL,First,NGUYEN THI KIM NGAN,Health issue
4,2026-02-27 00:00:00,3085158,AB,NaN,TA KHANH HOA,Family issue
...,...,...,...,...,...,...
874,2026-08-25 00:00:00,3109551,HAL,Second,HOANG DAI HAI,Health issue
875,2026-08-25 00:00:00,3112804,HAL,Second,NGUYEN THI THU THUY,Family issue
876,2026-08-25 00:00:00,3026462,AB,NaN,VO TIEN DAT,Health issue
877,2026-08-23 00:00:00,3091524,AB,NaN,LE NGUYEN DAN ANH,Unknown reason
